In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
from tool3_2 import visualize_model_architecture_text

MODEL_NAME = 'klue/bert-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model  = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [7]:
from tool3_2 import print_outputs0_text_visualization

sentence = '오늘 날씨가 정말 좋네요.'
inputs = tokenizer(sentence, return_tensors='pt')
with torch.no_grad():
    outputs = model(**inputs)

# outputs[0] and outputs[1] are tuple-style accessors
print('output keys:', list(outputs.keys()))
print('outputs[0] shape:', outputs[0].shape)
print('outputs[1] shape:', outputs[1].shape)

last_hidden_state = outputs[0]  # same as outputs.last_hidden_state
pooler_output = outputs[1]      # same as outputs.pooler_output
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print_outputs0_text_visualization(tokens, last_hidden_state)

cls_vector = last_hidden_state[0, 0, :]  # 
print(cls_vector)

output keys: ['last_hidden_state', 'pooler_output']
outputs[0] shape: torch.Size([1, 10, 768])
outputs[1] shape: torch.Size([1, 768])
=== Text visualization of outputs[0] ===
outputs[0] = last_hidden_state (batch, seq_len, hidden_size)
batch_size=1, seq_len=10, hidden_size=768
batch[0]
  token[ 0] [CLS]        -> hidden[768] [0.184, -1.110, 0.513, 1.567, ...]
  token[ 1] 오늘           -> hidden[768] [-0.722, -0.831, 0.216, 2.036, ...]
  token[ 2] 날씨           -> hidden[768] [-0.202, -0.074, 0.008, 1.743, ...]
  token[ 3] ##가          -> hidden[768] [-0.052, -0.455, -0.425, 2.354, ...]
  token[ 4] 정말           -> hidden[768] [0.414, 0.129, 0.060, 0.639, ...]
  token[ 5] 좋            -> hidden[768] [-0.455, -0.071, -0.717, 1.237, ...]
  token[ 6] ##네          -> hidden[768] [0.961, -0.440, -0.827, 1.179, ...]
  token[ 7] ##요          -> hidden[768] [0.221, 0.269, -1.137, 1.249, ...]
  token[ 8] .            -> hidden[768] [0.190, -1.170, 0.040, 0.785, ...]
  token[ 9] [SEP]        -> hidd

## BertPooler는 무엇인가?

`BertPooler`는 마지막 레이어의 `[CLS]` 벡터를 문장 레벨 표현으로 바꾸는 작은 변환층입니다.

- 입력: `last_hidden_state[:, 0, :]` (각 배치의 `[CLS]` 벡터)
- 연산: `Dense(768 -> 768) + Tanh`
- 출력: `pooler_output` (`outputs[1]`)

수식으로 쓰면 다음과 같습니다.

`pooler_output = tanh(W * CLS + b)`

여기서 `CLS = last_hidden_state[:, 0, :]` 입니다.

> 참고: BERT pretraining에서 pooler는 NSP(Next Sentence Prediction) 관련 신호를 함께 받았기 때문에, 단순한 mean pooling과는 성격이 다릅니다.

In [8]:
# Reproduce BertPooler output step by step
import torch.nn.functional as F

if not hasattr(model, 'pooler') or model.pooler is None:
    raise RuntimeError('This model does not have a pooler layer.')

cls_from_last_hidden = outputs.last_hidden_state[:, 0, :]  # [batch, hidden_size]
hf_pooler_output = outputs.pooler_output

# Manual formula: tanh(W * CLS + b)
W = model.pooler.dense.weight
b = model.pooler.dense.bias
manual_pooler_output = torch.tanh(F.linear(cls_from_last_hidden, W, b))

# Module call (same computation)
module_pooler_output = model.pooler(outputs.last_hidden_state)

max_abs_diff_manual = (manual_pooler_output - hf_pooler_output).abs().max().item()
max_abs_diff_module = (module_pooler_output - hf_pooler_output).abs().max().item()

In [13]:
sent1 = '나는 은행에 갔다'
sent2 = '강가에 있는 은행 나무'
# 1. tokenizer 를 사용해서 "은행" vector 를 뽑아낸다 <- 요건 최대한 직접 코드를 보면서 개발
#  - tokenizer(..., return_tensors='pt')
#  - tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
# with torch.no_grad():
#     outpus = model(**inputs)
#     ouputs.last_hidden_state[0, <은행 index 숫자값>, :].numpy()
# 2. 두개의 vector에 대한 cosine similirity 를 계산한다 <- AI 물어볼것

def get_token_embedding(sentence, target_word, tokenizer, model):
    """문장 속 target_word의 BERT 임베딩을 반환합니다."""
    inputs = tokenizer(sentence, return_tensors='pt')
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    with torch.no_grad():
        outputs = model(**inputs)
    # target_word를 포함하는 첫 토큰 위치 찾기
    for i, tok in enumerate(tokens):
        if target_word in tok:
            return outputs.last_hidden_state[0, i, :].numpy()
    return None

emb1 = get_token_embedding(sent1, '은행', tokenizer, model)
emb2 = get_token_embedding(sent2, '은행', tokenizer, model)

나는 은행에 갔다
tensor([   2,  717, 2259, 3924, 2170,  552, 2062,    3])
['[CLS]', '나', '##는', '은행', '##에', '갔', '##다', '[SEP]']
